In [1]:
from seeq import spy
spy.jobs.schedule("every day at 8:00 AM")

,Schedule,Scheduled,Next Run
0,every day at 8:00 AM,At 08:00 AM,2026-06-04 08:00:00 BRT


,Schedule,Scheduled,Next Run
0,every day at 8:00 AM,At 08:00 AM,2026-06-04 08:00:00 BRT


# anomaly_delay · Correção de age_risk e Ponderação por Qualidade

Estende o pipeline fb14-datalab com duas análises complementares:

1. **age_risk_corrected** — desconta do tempo de ciclo as horas em que a máquina
   esteve parada (`Tempo Delay > 0`), pois peças mecânicas não acumulam desgaste
   quando a linha para.

2. **quality_weight** — pondera eventos de cartão amarelo pelo delay coincidente,
   waste e ocorrência de falha registrada. Alertas durante paradas operacionais
   têm peso diagnóstico diferente de alertas em produção plena.

**Sinal-chave:** `BRSZ FB14 Quality - FB14_B1_Tempo Delay Calculad`  
**Workbook Seeq:** `0F116FED-4C4F-FD60-9EA0-0CA92EE4B765` (sheet FB14)  
**Output:** `anomaly_delay.csv` no diretório `notebooks/`  
**Pré-requisito:** `02_sinais_forca.csv` e `01_weibull_params.json` presentes.

## 1. Configuração

In [2]:
import sys
sys.path.insert(0, '..')

import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

# ── Parâmetros ─────────────────────────────────────────────────────────────
WORKBOOK_ID         = '0F116FED-4C4F-FD60-9EA0-0CA92EE4B765'
ANOS_HISTORICO      = 4
DELAY_THRESHOLD_MIN = 0       # delay > 0 min → máquina considerada parada
GRID_PULL           = '1h'    # compatível com frequência dos sinais de força
OUTPUT_CSV          = 'anomaly_delay.csv'

# Pesos de quality_weight
W_DELAY = 0.40
W_WASTE = 0.35
W_FAULT = 0.25

# Thresholds de cartão amarelo (mesmo critério do rca_fb14_fetch.ipynb)
THRESHOLD_P_RISK  = 0.40
THRESHOLD_FORCA_N = 800.0
JANELA_EVENTO_H   = 24.0   # ±2h em torno do cartão amarelo

# IDs dos sinais de força (de config.yaml)
SIGNAL_IDS_FORCA = {
    'Forca_A': '951157FA-D4BB-4696-A534-AEE4B48532CB',
    'Forca_B': '8D9E2FE1-6000-438C-B293-0EDDAA182851',
}

print('Configuração OK')
print(f'  Workbook  : {WORKBOOK_ID}')
print(f'  Histórico : {ANOS_HISTORICO} anos')
print(f'  Delay ≥   : {DELAY_THRESHOLD_MIN} min → parado')
print(f'  Pesos     : delay={W_DELAY}, waste={W_WASTE}, fault={W_FAULT}')

Configuração OK
  Workbook  : 0F116FED-4C4F-FD60-9EA0-0CA92EE4B765
  Histórico : 4 anos
  Delay ≥   : 0 min → parado
  Pesos     : delay=0.4, waste=0.35, fault=0.25


## 2. Buscar sinais de qualidade

Usa `spy.search` para descobrir os IDs reais dos sinais de delay, waste,
falha e uptime antes do pull. O sinal esperado é
`BRSZ FB14 Quality - FB14_B1_Tempo Delay Calculad`.

In [ ]:
QUALITY_GROUPS = {
    'delay':  ['*FB14*Delay*', '*FB14*Tempo Delay*', '*FB14_B1_Tempo*'],
    'waste':  ['*FB14*Waste*', '*FB14*Refugo*', '*FB14*NOK*'],
    'fault':  ['*FB14*Fault*', '*FB14*Falha*', '*FB14*Cod*Falha*', '*FB14*Parada*'],
    'uptime': ['*FB14*Uptime*', '*FB14*Disponib*', '*Bagger*FB14*', '*FB14*OEE*'],
}

encontrados = {}
todos = []

for grupo, patterns in QUALITY_GROUPS.items():
    resultados_grupo = []
    for pat in patterns:
        try:
            df_s = spy.search({'Name': pat}, quiet=True)
            if df_s is not None and not df_s.empty:
                df_s = df_s.copy()
                df_s['grupo'] = grupo
                df_s['pattern'] = pat
                resultados_grupo.append(df_s)
        except Exception as e:
            print(f'  [WARN] {pat}: {e}')

    if resultados_grupo:
        df_g = pd.concat(resultados_grupo, ignore_index=True).drop_duplicates(subset='ID')
        encontrados[grupo] = df_g
        todos.append(df_g)
        print(f'  {grupo:8s}: {len(df_g)} sinal(is)')
    else:
        encontrados[grupo] = pd.DataFrame()
        print(f'  {grupo:8s}: 0 sinais encontrados')

if todos:
    df_quality_all = pd.concat(todos, ignore_index=True).drop_duplicates(subset='ID')
    display_cols = [c for c in ['Name', 'Description', 'Value Unit Of Measure',
                                'Datasource Name', 'grupo', 'ID']
                   if c in df_quality_all.columns]
    print(f'\nTotal de sinais únicos encontrados: {len(df_quality_all)}')
    print(df_quality_all[display_cols].to_string())
else:
    print('Nenhum sinal de qualidade encontrado. Verifique conexão com Seeq.')
    df_quality_all = pd.DataFrame()

## 3. Pull: Tempo Delay Calculado

In [4]:
user_tz    = spy.session.get_user_timezone()
end_time   = pd.Timestamp.now(tz=user_tz)
start_time = end_time - pd.DateOffset(years=ANOS_HISTORICO)

print(f'Timezone : {user_tz}')
print(f'Período  : {start_time.date()} → {end_time.date()}')

delay_df = encontrados.get('delay', pd.DataFrame())

if delay_df.empty:
    print('⚠️  Sinal de delay não encontrado — correção de age desabilitada.')
    df_delay = None
else:
    mask_pref = delay_df['Name'].str.contains('Tempo Delay', case=False, na=False)
    delay_candidates = delay_df[mask_pref] if mask_pref.any() else delay_df

    if 'Type' in delay_candidates.columns:
        _mask_sig = delay_candidates['Type'].str.lower().str.contains('signal', na=False)
        delay_candidates = delay_candidates[_mask_sig]

    if delay_candidates.empty:
        print('⚠️  Nenhum sinal de tipo Signal encontrado para delay — correção desabilitada.')
        df_delay = None
    else:
        delay_items = pd.DataFrame([
            {'ID': row['ID'], 'Type': row.get('Type', 'Signal'), 'Name': row.get('Name', f"Signal_{i}")}
            for i, (_, row) in enumerate(delay_candidates.iterrows())
        ])

        print(f'Puxando {len(delay_items)} sinal(is) de delay...')
        dfs_ok = []
        for idx, row in delay_items.iterrows():
            try:
                df = spy.pull(
                    pd.DataFrame([{'ID': row['ID'], 'Type': row['Type']}]),
                    start=start_time.isoformat(),
                    end=end_time.isoformat(),
                    grid=GRID_PULL,
                    header='Name',
                    quiet=True,
                )
                if df is not None and not df.empty:
                    df.index = pd.to_datetime(df.index, utc=True)
                    dfs_ok.append(df)
                    print(f'  ✔️  Pull OK para: {row["Name"]}')
                else:
                    print(f'  ⚠️  Pull vazio para: {row["Name"]}')
            except Exception as e:
                print(f'  ❌ Erro ao puxar {row["Name"]}: {e}')

        if dfs_ok:
            df_delay = pd.concat(dfs_ok, axis=1)
            # Consolidar em coluna única 'delay_min' (média se múltiplos sinais)
            df_delay['delay_min'] = df_delay.select_dtypes(include='number').mean(axis=1)
            df_delay = df_delay[['delay_min']]
            print(f'Delay recebido: {len(df_delay):,} leituras')
            print(f'  delay_min > 0: {(df_delay["delay_min"] > DELAY_THRESHOLD_MIN).sum():,} horas paradas')
            print(f'  delay_min (p50/p95/max): '
                  f'{df_delay["delay_min"].quantile(0.50):.1f} / '
                  f'{df_delay["delay_min"].quantile(0.95):.1f} / '
                  f'{df_delay["delay_min"].max():.1f} min')
        else:
            print('⚠️  Nenhum delay válido retornado — correção de age desabilitada.')
            df_delay = None


Timezone : America/Sao_Paulo
Período  : 2022-06-03 → 2026-06-03
Puxando 18 sinal(is) de delay...
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_B2_Tempo Delay
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_B1_Tempo Delay Calculado
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_B1_Tempo Delay Calculado
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_Tempo Delay Calculado
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_B2_Tempo Delay Calculado
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_B1_Tempo Delay Calculado
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_Tempo Delay
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_Tempo Delay Calculado
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_Tempo Delay
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_B1_Tempo Delay
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_B1_Tempo Delay
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_B2_Tempo Delay
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_B1_Tempo Delay
  ✔️  Pull OK para: BRSZ FB14 Quality - FB14_Tempo Delay
  ✔️  Pull OK para

## 4. Pull: sinais de qualidade (waste, fault, uptime)

In [5]:
quality_pulls = {}

for grupo in ['waste', 'fault', 'uptime']:
    df_g = encontrados.get(grupo, pd.DataFrame())
    if df_g.empty:
        print(f'  {grupo:8s}: sem sinais → ignorado')
        quality_pulls[grupo] = None
        continue

    if 'Type' in df_g.columns:
        _mask_sig = df_g['Type'].str.lower().str.contains('signal', na=False)
        df_g = df_g[_mask_sig]

    if df_g.empty:
        print(f'  {grupo:8s}: nenhum Signal (apenas Conditions/Scalars) → ignorado')
        quality_pulls[grupo] = None
        continue

    items = pd.DataFrame([
        {'ID': r['ID'], 'Type': r.get('Type', 'Signal'), 'Name': r.get('Name', f'Signal_{i}')}
        for i, (_, r) in enumerate(df_g.iterrows())
    ])

    # Pull individual por sinal — mesmo padrão de cell-pull-delay
    dfs_ok = []
    for _, row in items.iterrows():
        try:
            df_p = spy.pull(
                pd.DataFrame([{'ID': row['ID'], 'Type': row['Type']}]),
                start=start_time.isoformat(),
                end=end_time.isoformat(),
                grid=GRID_PULL,
                header='Name',
                quiet=True,
            )
            if df_p is not None and not df_p.empty:
                df_p.index = pd.to_datetime(df_p.index, utc=True)
                dfs_ok.append(df_p)
        except Exception as e:
            print(f'  [{grupo}] ❌ {row["Name"]}: {e}')

    if dfs_ok:
        quality_pulls[grupo] = pd.concat(dfs_ok, axis=1)
        print(f'  {grupo:8s}: {len(quality_pulls[grupo]):,} leituras  |  {len(dfs_ok)} sinal(is)')
    else:
        quality_pulls[grupo] = None
        print(f'  {grupo:8s}: pull vazio ou todos os sinais falharam')

  [waste] ❌ BRSZ_AF_Centerline_FB14__GPD Flap JFold.Parameter A - NOK: (503)
Reason: Service Unavailable
HTTP response headers: HTTPHeaderDict({'Date': 'Wed, 03 Jun 2026 14:44:34 GMT', 'Content-Type': 'application/vnd.seeq.v1+json', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Server-Timing': '0;dur=548.42;desc=Datasource,1;dur=0.00;desc=Cache,2;dur=252.03;desc=Processing,3;dur=0.00;desc=GC,4;dur=0.09;desc="Request Queue",5;dur=0.00;desc="Calc Engine Queue",6;dur=2.05;desc="Seeq Database",7;dur=802.58;desc=Total', 'Server-Meters': '0;dur=0;desc="Datasource Samples Read",1;dur=0;desc="Datasource Capsules Read",2;dur=0;desc="Cache Samples Read",3;dur=0;desc="Cache Capsules Read",4;dur=0;desc="Cache In-Memory Samples Read",5;dur=0;desc="Cache In-Memory Capsules Read",6;dur=3;desc="Database Items Read",7;dur=0;desc="Database Relationships Read"', 'x-sq-request-id': 'R^a2BSpG46RbqlalZvxMC3bA', 'Server': 'Seeq/R66.144.0-v202605272039-CDH', 'Content-Security-Policy': "default-

## 5. Carregar dados de força (02_sinais_forca.csv)

Reutiliza o CSV pré-computado sem re-pull do Seeq — mesma decisão do
`rca_fb14_fetch.ipynb`. Apenas leituras com `ciclo_id >= 0` têm
`horas_desde_troca` e features calculadas.

In [6]:
df_forca = pd.read_csv('02_sinais_forca.csv', parse_dates=['Timestamp'])
df_forca['Timestamp'] = pd.to_datetime(df_forca['Timestamp'], utc=True)
df_forca = df_forca.sort_values('Timestamp').reset_index(drop=True)

df_ciclos = df_forca[df_forca['ciclo_id'] >= 0].copy()

print(f'Total de leituras          : {len(df_forca):,}')
print(f'Leituras com ciclo_id >= 0 : {len(df_ciclos):,}')
print(f'Ciclos únicos              : {df_ciclos["ciclo_id"].nunique()}')
print(f'Período                    : {df_ciclos["Timestamp"].min()} → {df_ciclos["Timestamp"].max()}')

Total de leituras          : 3,281
Leituras com ciclo_id >= 0 : 1,224
Ciclos únicos              : 16
Período                    : 2025-02-18 15:04:03+00:00 → 2026-06-02 23:00:29+00:00


## 6. Corrigir age_effective por ciclo

Para cada leitura com `ciclo_id >= 0`:

```
stopped_h       = Σ horas com delay_min > 0 desde o início do ciclo até o timestamp
age_effective_h = horas_desde_troca − stopped_h  (mín 0)
age_risk_corr   = 1 − exp(−(age_effective_h / η)^β)
```

In [7]:
with open('01_weibull_params.json') as f:
    wp = json.load(f)
beta = wp['weibull_beta']
eta  = wp['weibull_eta_h']
print(f'Weibull: β={beta}, η={eta}h')

df_ciclos = df_ciclos.copy()
df_ciclos['age_risk_original'] = df_ciclos['age_risk']

stopped_h_list     = []
age_eff_list       = []
age_risk_corr_list = []

for ciclo_id_val, grp in df_ciclos.groupby('ciclo_id'):
    ciclo_start = grp['Timestamp'].min()

    for _, row in grp.iterrows():
        if df_delay is not None:
            mask = (df_delay.index >= ciclo_start) & (df_delay.index <= row['Timestamp'])
            # grid=1h → cada linha representa 1 hora
            stopped_h = float((df_delay.loc[mask, 'delay_min'] > DELAY_THRESHOLD_MIN).sum())
        else:
            stopped_h = 0.0

        age_cal   = row['horas_desde_troca'] if pd.notna(row['horas_desde_troca']) else 0.0
        age_eff   = max(0.0, float(age_cal) - stopped_h)
        age_rc    = 1.0 - np.exp(-(age_eff / eta) ** beta)

        stopped_h_list.append(stopped_h)
        age_eff_list.append(age_eff)
        age_risk_corr_list.append(age_rc)

df_ciclos['stopped_h']          = stopped_h_list
df_ciclos['age_effective_h']    = age_eff_list
df_ciclos['age_risk_corrected'] = age_risk_corr_list

delta = df_ciclos['age_risk_original'] - df_ciclos['age_risk_corrected']
print(f'\nDelta age_risk (original − corrigido):')
print(delta.describe().to_string())
print(f'\nLeituras com parada acumulada > 0h : {(df_ciclos["stopped_h"] > 0).sum():,}')

Weibull: β=1.397762, η=860.61h

Delta age_risk (original − corrigido):
count    1224.000000
mean        0.036489
std         0.038092
min        -0.024096
25%         0.008630
50%         0.031512
75%         0.049584
max         0.134286

Leituras com parada acumulada > 0h : 1,207


## 7. Calcular quality_weight por evento de cartão amarelo

```
delay_weight   = clip(Σ delay_min na janela ±2h  / p95_global_delay,  0, 1)
waste_weight   = clip(Σ waste na janela ±2h       / p95_global_waste,  0, 1)
fault_weight   = 1.0 se algum fault_code > 0 na janela ±2h, senão 0.0
quality_weight = 0.40×delay + 0.35×waste + 0.25×fault
```

Leituras sem cartão amarelo recebem `quality_weight = 0`.

In [8]:
def p95_of_pull(df_pull):
    if df_pull is None:
        return 1.0
    vals = df_pull.select_dtypes(include='number').values.flatten()
    vals = vals[~np.isnan(vals)]
    return float(np.percentile(vals, 95)) if len(vals) > 0 else 1.0

p95_delay = p95_of_pull(df_delay)
p95_waste = p95_of_pull(quality_pulls.get('waste'))

print(f'p95 delay : {p95_delay:.2f} min')
print(f'p95 waste : {p95_waste:.4f}')

janela_td    = pd.Timedelta(hours=JANELA_EVENTO_H)
q_waste_pull = quality_pulls.get('waste')
q_fault_pull = quality_pulls.get('fault')

mask_yellow = (
    (df_ciclos['p_risk'] > THRESHOLD_P_RISK) |
    (df_ciclos['min_forca_3d'] < THRESHOLD_FORCA_N)
)
df_ciclos['is_yellow_card'] = mask_yellow
print(f'\nCartões amarelos: {mask_yellow.sum():,}')

dw_list, ww_list, fw_list, qw_list = [], [], [], []

for _, row in df_ciclos.iterrows():
    if not row['is_yellow_card']:
        dw_list.append(0.0); ww_list.append(0.0)
        fw_list.append(0.0); qw_list.append(0.0)
        continue

    ts = row['Timestamp']
    t0, t1 = ts - janela_td, ts + janela_td

    # delay weight
    if df_delay is not None:
        delay_sum = float(df_delay.loc[t0:t1, 'delay_min'].sum())
        dw = float(np.clip(delay_sum / max(p95_delay, 1e-6), 0, 1))
    else:
        dw = 0.0

    # waste weight
    if q_waste_pull is not None:
        waste_vals = q_waste_pull.loc[t0:t1].select_dtypes(include='number').values.flatten()
        ww = float(np.clip(np.nansum(waste_vals) / max(p95_waste, 1e-6), 0, 1))
    else:
        ww = 0.0

    # fault weight
    fw = 0.0
    if q_fault_pull is not None:
        fault_vals = q_fault_pull.loc[t0:t1].select_dtypes(include='number').values.flatten()
        if np.any(fault_vals > 0):
            fw = 1.0

    qw = W_DELAY * dw + W_WASTE * ww + W_FAULT * fw
    dw_list.append(dw); ww_list.append(ww)
    fw_list.append(fw); qw_list.append(qw)

df_ciclos['delay_weight']   = dw_list
df_ciclos['waste_weight']   = ww_list
df_ciclos['fault_weight']   = fw_list
df_ciclos['quality_weight'] = qw_list

amarelos = df_ciclos[df_ciclos['is_yellow_card']]
print(f'\nQuality weight (cartões amarelos):')
print(amarelos['quality_weight'].describe().to_string())

p95 delay : 305.33 min
p95 waste : 123.6700

Cartões amarelos: 703

Quality weight (cartões amarelos):
count    703.000000
mean       0.993816
std        0.048943
min        0.600000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000


## 8. Exportar anomaly_delay.csv

In [9]:
COLS_EXPORT = [
    'Timestamp', 'ciclo_id',
    'horas_desde_troca',    # age calendário (horas)
    'stopped_h',            # horas paradas acumuladas no ciclo
    'age_effective_h',      # horas_desde_troca − stopped_h
    'age_risk_original',    # Weibull sem correção de parada
    'age_risk_corrected',   # Weibull com age_effective
    'p_risk',
    'min_forca_3d',
    'is_yellow_card',
    'delay_weight',
    'waste_weight',
    'fault_weight',
    'quality_weight',
]

cols_presentes = [c for c in COLS_EXPORT if c in df_ciclos.columns]
df_out = df_ciclos[cols_presentes].copy()

df_out.to_csv(OUTPUT_CSV, index=False)

print(f'Exportado: {OUTPUT_CSV}')
print(f'  Linhas           : {len(df_out):,}')
print(f'  Cartões amarelos : {df_out["is_yellow_card"].sum():,}')
print(f'  Com parada > 0h  : {(df_out["stopped_h"] > 0).sum():,}')

sample = df_out[df_out['is_yellow_card'] & (df_out['quality_weight'] > 0)]
if not sample.empty:
    print(f'\nCartões amarelos com quality_weight > 0 ({len(sample)}):')    
    print(sample[['Timestamp', 'ciclo_id', 'age_risk_original',
                  'age_risk_corrected', 'quality_weight']].to_string())

Exportado: anomaly_delay.csv
  Linhas           : 1,224
  Cartões amarelos : 703
  Com parada > 0h  : 1,207

Cartões amarelos com quality_weight > 0 (703):
                     Timestamp  ciclo_id  age_risk_original  age_risk_corrected  quality_weight
2058 2025-02-18 23:05:48+00:00         0           0.008553            0.005965        1.000000
2059 2025-02-19 07:05:27+00:00         0           0.012127            0.008332        1.000000
2060 2025-02-19 15:01:16+00:00         0           0.015830            0.011323        1.000000
2061 2025-02-19 22:58:08+00:00         0           0.019666            0.014080        1.000000
2062 2025-02-20 06:42:16+00:00         0           0.023500            0.015891        1.000000
2063 2025-02-20 15:05:37+00:00         0           0.027753            0.019122        1.000000
2064 2025-02-20 22:57:39+00:00         0           0.031818            0.021154        1.000000
2099 2025-03-12 08:41:26+00:00         1           0.160668            0.114